In [2]:
import os
import shutil
import re

#### Move batch files to data folder

In [3]:
data_folder = 'data'
os.makedirs(data_folder, exist_ok=True)

for root, dirs, files in os.walk('.'):
    for dir_name in dirs:
        if 'batch' in dir_name:
            batch_folder_path = os.path.join(root, dir_name)
            for file_name in os.listdir(batch_folder_path):
                if file_name.endswith('.txt') or file_name.endswith('.ann'):
                    file_path = os.path.join(batch_folder_path, file_name)
                    shutil.move(file_path, data_folder)
                    print(f"Moved {file_path} to {data_folder}")

#### Move .ann Data

In [4]:

data_folder = 'corpora/data'
os.makedirs(data_folder, exist_ok=True)
lct_txt = 'corpora/lct_ann'

for file_name in os.listdir(data_folder):
    if file_name.endswith('.ann'):
        file_path = os.path.join(data_folder, file_name)
        shutil.copy(file_path, lct_txt)
        print(f"Moved {file_path} to {lct_txt}")

## Splitte Inc und Exc aus allen Daten

In [5]:


def read_criteria_file(file_path):
    with open(file_path, 'r', encoding='utf-8') as file:
        return file.read()

def write_to_file(file_path, content):
    with open(file_path, 'w', encoding='utf-8') as file:
        file.write(content)

def process_files(input_folder, output_folder):
    os.makedirs(output_folder, exist_ok=True)
    unsplit_files = []

    for filename in os.listdir(input_folder):
        if filename.endswith(".txt"):
            nct_number = re.findall(r'NCT\d+', filename)
            if not nct_number:
                nct_number = re.findall(r'nct\d+', filename, re.IGNORECASE)
                if not nct_number:
                    unsplit_files.append(filename)
                    continue
            nct_number = nct_number[0]

            file_path = os.path.join(input_folder, filename)
            content = read_criteria_file(file_path)
            lines = content.split('\n')

            inclusion_criteria = []
            exclusion_criteria = []
            current_section = None

            for line in lines:
                line_lower = line.lower().strip()
                if "inclusion criteria" in line_lower:
                    current_section = "inclusion"
                    continue
                elif "exclusion criteria" in line_lower:
                    current_section = "exclusion"
                    continue

                if current_section == "inclusion":
                    if line.strip().startswith("-") or re.match(r'^\s*\d+\.\s', line.strip()):
                        inclusion_criteria.append(line.strip())
                    elif line.strip() and inclusion_criteria:
                        inclusion_criteria[-1] += ' ' + line.strip()
                elif current_section == "exclusion":
                    if line.strip().startswith("-") or re.match(r'^\s*\d+\.\s', line.strip()):
                        exclusion_criteria.append(line.strip())
                    elif line.strip() and exclusion_criteria:
                        exclusion_criteria[-1] += ' ' + line.strip()

            # Remove empty lines and duplicates
            inclusion_criteria = list(dict.fromkeys([line for line in inclusion_criteria if line]))
            exclusion_criteria = list(dict.fromkeys([line for line in exclusion_criteria if line]))

            if inclusion_criteria:
                inc_filename = f"{nct_number}_inc.txt"
                inc_file_path = os.path.join(output_folder, inc_filename)
                write_to_file(inc_file_path, "\n".join(inclusion_criteria))

            if exclusion_criteria:
                exc_filename = f"{nct_number}_exc.txt"
                exc_file_path = os.path.join(output_folder, exc_filename)
                write_to_file(exc_file_path, "\n".join(exclusion_criteria))

    # Print out the filenames that couldn't be split into inc and exc
    if unsplit_files:
        print("Files that couldn't be split into inclusion and exclusion criteria:")
        for file in unsplit_files:
            print(file)

# Beispielaufruf
input_folder = "corpora/lct_p1"
output_folder = "corpora/lct_p1_half"
process_files(input_folder, output_folder)

In [6]:
# 986 Dateien
# ec und ic -> 
# NCT03860181

## Zähle Operatoren

In [1]:
import os
import json

def count_operators(directory):
    and_count = 0
    or_count = 0
    not_count = 0

    json_files = [f for f in os.listdir(directory) if f.endswith('.json')]

    for file in json_files:
        file_path = os.path.join(directory, file)
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
            for entry in data:
                if entry['type'] == 'And':
                    and_count += 1
                elif entry['type'] == 'Or':
                    or_count += 1
                elif entry['type'] == 'Negation':
                    not_count += 1

    return and_count, or_count, not_count

directory = 'all_entitys'
and_count, or_count, not_count = count_operators(directory)

print(f"Total AND operators: {and_count}")
print(f"Total NOT operators: {not_count}")
print(f"Total OR operators: {or_count}")



#### Replace double spaces

In [ ]:
def replace_double_spaces(folder):
    for filename in os.listdir(folder):
        file_path = os.path.join(folder, filename)
        if os.path.isfile(file_path):
            with open(file_path, 'r', encoding='utf-8') as f:
                content = f.read()

            content = re.sub(r'  +', ' ', content)

            print(content)
            with open(file_path, 'w', encoding='utf-8') as f:
                f.write(content)